# guppy and hugr to qir conversion and submission to H2


This shows:
- Use of constants at comptime
- Use of comptime arrays and python functions on them
- Conversion of boolean result array to integer


In [13]:
from typing import no_type_check

from guppylang import guppy, qubit
from guppylang.std.builtins import result
from guppylang.std.quantum import h, cx, x, measure

N = 10

# pure python function
def generate_ghz(qbs: list[qubit]) -> None:
    if len(qbs) == 0:
        return
    h(qbs[0])
    for i in range(len(qbs) - 1):
        cx(qbs[i], qbs[i+1])

# bool array to int (big endian)
def to_int(result_array: list[bool]) -> int:
    result = 0
    for b in result_array:
        # Shift left by 1 (multiply by 2) and add 1 if True, else 0
        result = (result << 1) | int(b)
    return result

@guppy.comptime
@no_type_check
def main() -> None:
    comptime_array = [qubit() for _ in range(N)]
    generate_ghz(comptime_array)
    for q in comptime_array:
        x(q)
    results = [measure(q) for q in comptime_array]
    result("encoded_results", to_int(results))

This shows:
- breakout to guppy func for measure dependent action
- @owned decorator


In [10]:
from typing import no_type_check

from guppylang import guppy, qubit
from guppylang.std.builtins import owned, result
from guppylang.std.quantum import h, cx, x, measure

@guppy
def teleport(alice: qubit@owned, bob: qubit) -> None:
    if measure(alice):
        x(bob)

@guppy.comptime
@no_type_check
def main() -> None:
    alice, bob = qubit(), qubit()

    h(alice)
    cx(alice, bob)

    teleport(alice, bob)

    result("bob", measure(bob))

# Convert hugr to qir

By default, the function will automatically check the generated QIR to capture most of the issues that could happen.
This will show an error message with more details about the problem the check can be turned off using the keyword argument `validate_qir = False`

In [14]:
from hugr_qir.guppy_to_qir import guppy_to_qir_str, guppy_to_qir_bytes
guppy_qir_bitcode_string = guppy_to_qir_str(main)

In [15]:
# To get a human-readable LLVM assemly language string use the `hugr_to_qir` function with the keyword argument `output_format = OutputFormat.LLVM_IR`
from hugr_qir.guppy_to_qir import guppy_to_qir_str, guppy_to_qir_bytes

guppy_qir = guppy_to_qir_str(main)
print(guppy_qir)

; ModuleID = 'hugr-qir'
source_filename = "hugr-qir"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i8:8:32-i16:16:32-i64:64-i128:128-n32:64-S128-Fn32"
target triple = "aarch64-unknown-linux-gnu"

@0 = private unnamed_addr constant [16 x i8] c"encoded_results\00", align 1
@gen_name = private unnamed_addr constant [8 x i8] c"hugr-qir", section ",qir_generator"
@gen_version = private unnamed_addr constant [10 x i8] c"0.2.0-rc.1", section ",qir_generator"

define void @__hugr__.main.1() local_unnamed_addr #0 {
alloca_block:
  tail call void @__quantum__rt__initialize(ptr null)
  tail call void @__quantum__qis__phasedx__body(double 0x3FF921FB54442D18, double 0xBFF921FB54442D18, ptr null)
  tail call void @__quantum__qis__rz__body(double 0x400921FB54442D18, ptr null)
  tail call void @__quantum__qis__phasedx__body(double 0xBFF921FB54442D18, double 0x3FF921FB54442D18, ptr nonnull inttoptr (i64 1 to ptr))
  tail call void @__quantum__qis__rzz__body(double 0x3FF921FB54442D18, ptr 

### Loops in the program are unrolled automatically when possible, because backwards branching is not available on H Series.  This means that the QIR generated from programs containing loops can get quite long as shown here:

# Submission to the device via Nexus

The QIR generated can be submitted directly to Nexus. The python Nexus API is available via `pip install qnexus`. This requires a different QIR format for the submission, which can be generated from `compile_qir`.

In [22]:
import qnexus as qnx

qnx.login()

In [14]:
import datetime

project = qnx.projects.get_or_create(name="QIR-Demonstration3")
qnx.context.set_active_project(project)

qir_name = "HUGR-QIR"
jobname_suffix = datetime.datetime.now().strftime("%Y_%m_%d-%H-%M-%S")

In [15]:
# You can write your guppy directly in a notebook or in a separate file
from typing import no_type_check

from guppylang import guppy


@guppy
@no_type_check
def main() -> None:
    q0 = qubit()
    q1 = qubit()

    h(q0)
    h(q1)

    b0 = measure(q0)
    b1 = measure(q1)
    b2 = b0 ^ b1

    result("0", b2)

In [16]:
guppy_qir_bitcode = guppy_to_qir_bytes(main)

In [17]:
qir_program_ref = qnx.qir.upload(qir=guppy_qir_bitcode, name=qir_name, project=project)

In [18]:
# Run on the H2-1 Syntax checker
device_name = "H2-1SC"

qnx.context.set_active_project(project)
config = qnx.QuantinuumConfig(device_name=device_name)

job_name = f"execution-job-qir-{qir_name}-{device_name}-{jobname_suffix}"
ref_execute_job = qnx.start_execute_job(
    programs=[qir_program_ref],
    n_shots=[10],
    backend_config=config,
    name=job_name,
)

In [21]:
qnx.jobs.wait_for(ref_execute_job)

In [20]:
qir_result = qnx.jobs.results(ref_execute_job)[0].download_result()
qir_result.get_counts()

Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `%`, will omit from BackendInfo. Consider updating your pytket version.


Counter({(0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0): 10})